# Exercise 4 - CNN classification

> **GPU: Runtime -> Change runtime type -> T4 GPU.** Total runtime about 6 minutes on a T4.

We use **FashionMNIST** here rather than CIFAR-10, so this is not a copy of the lesson: 28x28
grayscale, 10 clothing classes, 1 input channel. Small differences that force you to actually
think about shapes and about which augmentations are valid.

Seven tasks.

In [ ]:
import math, time, sys
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = device.type == 'cuda'
if not USE_AMP:
    print('*** no GPU: reduce EPOCHS below, or switch the runtime ***')
print('device', device)

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
DATA_DIR = '/content/data' if IN_COLAB else './data'
NUM_WORKERS = 2 if IN_COLAB else 0

def set_seed(seed=0):
    import random
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

set_seed(0)
torch.backends.cudnn.benchmark = True
plt.rcParams['figure.dpi'] = 110

FM_MEAN, FM_STD = (0.2860,), (0.3530,)      # FashionMNIST train-split stats, 1 channel
raw_train = datasets.FashionMNIST(DATA_DIR, train=True, download=True)
raw_val = datasets.FashionMNIST(DATA_DIR, train=False, download=True)
CLASSES = raw_train.classes
print('classes:', CLASSES)
print('train', len(raw_train), '| val', len(raw_val), '| image', raw_train[0][0].size, raw_train[0][0].mode)

---
## Task 1 - Two transform pipelines

Build `train_tf` and `eval_tf` for FashionMNIST:

- **both**: to tensor, normalized with `FM_MEAN` / `FM_STD`
- **train only**: random crop 28 with padding 2, and a small random rotation (up to 10 degrees)
- **neither**: a horizontal flip. Think about why before you read the hint.

*Hint: a mirrored sandal is still a sandal, but a mirrored "2" is not a 2. FashionMNIST is
borderline - flip is actually acceptable here. The transform you must NOT use is a **vertical**
flip: an upside-down shirt is not something you will ever be asked to classify. Justify your
choice in the markdown cell below.*

In [ ]:
# TODO: define train_tf and eval_tf as transforms.Compose([...])

train_ds = datasets.FashionMNIST(DATA_DIR, train=True, transform=train_tf)
val_ds = datasets.FashionMNIST(DATA_DIR, train=False, transform=eval_tf)

x0, y0 = train_ds[0]
xe, _ = val_ds[0]
assert x0.shape == (1, 28, 28), f'train sample shape {tuple(x0.shape)} should be (1, 28, 28)'
assert xe.shape == (1, 28, 28), f'val sample shape {tuple(xe.shape)}'
assert x0.dtype == torch.float32
assert abs(float(xe.mean())) < 1.0 and float(xe.std()) > 0.5, 'val images do not look normalized'

names_train = [type(t).__name__ for t in train_tf.transforms]
names_eval = [type(t).__name__ for t in eval_tf.transforms]
assert 'Normalize' in names_train and 'Normalize' in names_eval, 'both pipelines must normalize'
assert names_eval == ['ToTensor', 'Normalize'], f'eval_tf must be exactly ToTensor+Normalize, got {names_eval}'
assert any('Random' in n for n in names_train), 'train_tf needs augmentation'
assert not any('Random' in n for n in names_eval), 'NEVER augment the validation set'
assert names_train.index('ToTensor') > max(i for i, n in enumerate(names_train) if 'Random' in n), \
    'PIL-based random transforms must come BEFORE ToTensor'
print('PASS  train:', names_train)
print('      eval :', names_eval)

**Which augmentations did you pick, and why?** ...

---
## Task 2 - Loaders, and look at a batch

Create `train_loader` (batch 128, shuffled, `drop_last=True`) and `val_loader` (batch 256, not
shuffled). Then write `denormalize` and plot 16 training images with their labels.

Look at the plot. If the labels don't match the pictures, stop and fix it - everything
downstream is meaningless.

In [ ]:
# TODO: train_loader, val_loader

def denormalize(t, mean=FM_MEAN, std=FM_STD):
    """(1, H, W) normalized tensor -> (H, W) numpy array in 0..1 for imshow."""
    # TODO
    raise NotImplementedError


assert train_loader.batch_size == 128 and val_loader.batch_size == 256
assert train_loader.drop_last and not val_loader.drop_last
assert isinstance(train_loader.sampler, torch.utils.data.RandomSampler), 'train_loader must shuffle'
assert isinstance(val_loader.sampler, torch.utils.data.SequentialSampler), 'val_loader must NOT shuffle'
assert len(train_loader) == 60000 // 128, f'{len(train_loader)} batches - is drop_last set?'

xb, yb = next(iter(train_loader))
assert xb.shape == (128, 1, 28, 28) and yb.dtype == torch.int64
d = denormalize(xb[0])
assert np.asarray(d).shape == (28, 28), f'denormalize should give (28, 28), got {np.asarray(d).shape}'
assert 0.0 <= np.asarray(d).min() and np.asarray(d).max() <= 1.0, 'denormalized values must be in 0..1'
print(f'PASS  batch {tuple(xb.shape)} | mean {xb.mean():+.3f} std {xb.std():.3f}')

fig, axes = plt.subplots(2, 8, figsize=(13, 3.6))
for ax, i in zip(axes.ravel(), range(16)):
    ax.imshow(denormalize(xb[i]), cmap='gray')
    ax.set_title(CLASSES[yb[i]], fontsize=7)
    ax.axis('off')
plt.suptitle('training batch after transforms - do the labels match?')
plt.tight_layout()

---
## Task 3 - The model

Build `FashionCNN` for **1-channel 28x28** input:

- three stages, each `Conv-BN-ReLU` then `MaxPool2d(2)`: 28 -> 14 -> 7 -> 3
- channels 1 -> 24 -> 48 -> 96
- head: global average pooling -> flatten -> dropout(0.2) -> linear to 10
- convs have `bias=False`
- under 120,000 parameters

In [ ]:
class FashionCNN(nn.Module):
    def __init__(self, n_classes=10, c_in=1, p_drop=0.2):
        super().__init__()
        # TODO
        raise NotImplementedError

    def forward(self, x):
        # TODO
        raise NotImplementedError


model = FashionCNN().to(device)
n_params = sum(p.numel() for p in model.parameters())
out = model(torch.randn(4, 1, 28, 28, device=device))

assert out.shape == (4, 10), f'output {tuple(out.shape)}'
assert n_params < 120_000, f'{n_params:,} parameters is too many'
assert sum(1 for m in model.modules() if isinstance(m, nn.BatchNorm2d)) >= 3, 'use BatchNorm'
assert all(m.bias is None for m in model.modules() if isinstance(m, nn.Conv2d)), 'convs need bias=False'
assert sum(1 for m in model.modules() if isinstance(m, nn.MaxPool2d)) == 3, 'exactly 3 pooling steps'
assert model(torch.randn(1, 1, 56, 56, device=device)).shape == (1, 10), 'head must be size-agnostic (use GAP)'
print(f'PASS  {n_params:,} parameters')

---
## Task 4 - Sanity checks

Two checks, before any real training.

1. **Initial loss.** An untrained 10-class model should score about `ln(10)`. Assert it's within
   0.3.
2. **Overfit one batch.** 8 samples, dropout off, train to loss < 0.01. If this fails there is
   a bug, and no hyperparameter will fix it.

In [ ]:
criterion = nn.CrossEntropyLoss()

def initial_loss(batch_x, batch_y):
    """Loss of a FRESH, untrained FashionCNN on one batch. Use .eval() and torch.no_grad()."""
    # TODO
    raise NotImplementedError


init_loss = initial_loss(xb.to(device), yb.to(device))
print(f'ln(10) = {math.log(10):.4f} | measured {init_loss:.4f}')
assert abs(init_loss - math.log(10)) < 0.3, f'initial loss {init_loss:.3f} is suspicious'
print('PASS  initial loss is as expected')

In [ ]:
def overfit_one_batch(xs, ys, steps=250, lr=1e-3):
    """Train a fresh dropout-free FashionCNN on `xs`/`ys` only. Return the list of losses."""
    # TODO
    raise NotImplementedError


xs, ys = xb[:8].to(device), yb[:8].to(device)
losses = overfit_one_batch(xs, ys)

assert losses[-1] < 0.01, f'final loss {losses[-1]:.4f} - cannot memorise 8 samples, so there is a BUG'
assert losses[-1] < losses[0] / 100, 'loss barely moved'
print(f'PASS  loss {losses[0]:.4f} -> {losses[-1]:.2e}: the pipeline is wired correctly')

plt.figure(figsize=(5, 3))
plt.plot(losses); plt.yscale('log'); plt.xlabel('step'); plt.ylabel('loss'); plt.grid(alpha=0.3)
plt.title('overfitting 8 samples')

---
## Task 5 - Train and evaluate functions

Write them properly. The assertions check the details that are easy to skip.

`train_one_epoch` must: set train mode, move data to device, zero gradients, forward, loss,
backward, step, and return `(mean_loss_per_sample, accuracy)`.

`evaluate` must: set eval mode, run under `no_grad`, and return the same pair - plus optional
predictions.

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device, scheduler=None):
    """-> (mean loss per SAMPLE, accuracy)"""
    # TODO
    raise NotImplementedError


@torch.no_grad()
def evaluate(model, loader, criterion, device, return_preds=False):
    """-> (loss, acc) or (loss, acc, y_true, y_pred, y_prob)"""
    # TODO
    raise NotImplementedError


set_seed(0)
probe = FashionCNN().to(device)
opt = torch.optim.SGD(probe.parameters(), lr=0.05, momentum=0.9)
small_loader = DataLoader(Subset(train_ds, range(2048)), batch_size=128, shuffle=True, drop_last=True)

probe.train()
tl, ta = train_one_epoch(probe, small_loader, criterion, opt, device)
assert 0.0 < tl < 5.0 and 0.0 <= ta <= 1.0, f'train_one_epoch returned {(tl, ta)}'
assert probe.training, 'model should be left in train mode by train_one_epoch'

vl, va = evaluate(probe, val_loader, criterion, device)
assert not probe.training, 'evaluate must call model.eval()'
assert 0.0 < vl < 5.0 and 0.0 <= va <= 1.0
res = evaluate(probe, val_loader, criterion, device, return_preds=True)
assert len(res) == 5, 'return_preds=True should return 5 things'
_, _, yt, yp, pr = res
assert yt.shape == yp.shape == (len(val_ds),), f'{yt.shape} {yp.shape}'
assert pr.shape == (len(val_ds), 10) and np.allclose(pr.sum(1), 1.0, atol=1e-4), 'y_prob rows must sum to 1'
assert abs((yp == yt).mean() - va) < 1e-6, 'accuracy and predictions disagree'
print(f'PASS  train ({tl:.4f}, {ta:.4f}) | val ({vl:.4f}, {va:.4f})')

---
## Task 6 - Train it

Train for `EPOCHS` with:

- SGD, lr 0.05, momentum 0.9, nesterov, weight decay 5e-4 **excluded from BatchNorm params and
  biases** (parameter groups)
- `CosineAnnealingLR` over the run
- track history and keep the best validation accuracy

Target: **val accuracy > 0.90** in 8 epochs. (A good FashionMNIST CNN reaches ~0.93.)

In [ ]:
EPOCHS = 8

set_seed(0)
model = FashionCNN().to(device)

# TODO: build the two parameter groups, the optimizer, the scheduler
# TODO: the epoch loop, filling history and tracking best_acc

history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_acc = 0.0
raise NotImplementedError

assert len(history['val_acc']) == EPOCHS
assert best_acc > 0.90, f'best val accuracy {best_acc:.4f} - below target 0.90'
assert history['train_loss'][-1] < history['train_loss'][0], 'training loss should decrease'
print(f'PASS  best val accuracy {best_acc:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
axes[0].plot(history['train_loss'], marker='.', label='train')
axes[0].plot(history['val_loss'], marker='.', label='val')
axes[0].set_ylabel('loss')
axes[1].plot(history['train_acc'], marker='.', label='train')
axes[1].plot(history['val_acc'], marker='.', label='val')
axes[1].set_ylabel('accuracy')
for ax in axes:
    ax.set_xlabel('epoch'); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()

---
## Task 7 - Evaluate beyond one number

Produce:

1. A confusion matrix and per-class recall. Which class is worst? Which pair is most confused?
2. The 12 most confident mistakes, plotted with true and predicted labels.

Then answer: what would you *do* about the worst class?

In [ ]:
def confusion_matrix(y_true, y_pred, k=10):
    # TODO
    raise NotImplementedError


_, final_acc, y_true, y_pred, y_prob = evaluate(model, val_loader, criterion, device, return_preds=True)
cm = confusion_matrix(y_true, y_pred)
recall = np.diag(cm) / cm.sum(1)

assert cm.shape == (10, 10) and cm.sum() == len(y_true)
assert abs(np.diag(cm).sum() / cm.sum() - final_acc) < 1e-6
print(f'accuracy {final_acc:.4f}')
for i, c in enumerate(CLASSES):
    print(f'  {c:14} recall {recall[i]:.3f}  support {cm[i].sum()}')
print(f'\nworst class: {CLASSES[recall.argmin()]} at {recall.min():.3f}')

off = cm.copy(); np.fill_diagonal(off, 0)
i, j = np.unravel_index(off.argmax(), off.shape)
print(f'most confused: true {CLASSES[i]} predicted as {CLASSES[j]} ({off[i, j]} times)')

# TODO: plot the 12 most confident mistakes with true/pred labels in the titles

**What would you do about the worst class?** ...

---
## Done

- [ ] I check the initial loss and overfit one batch before every real training run.
- [ ] I know which transforms belong on train only, and why.
- [ ] I can write the training loop with no reference.
- [ ] I never report only accuracy.

Solutions: [`solutions/sol04_cnn.ipynb`](solutions/sol04_cnn.ipynb)